# 04 · QAOA 与 VQE：变分算法

对应讲义实验 C · 前置：01 编程基础 · 另有课后作业

**问题**：Max-Cut。把图的顶点分成两组，让跨组的边尽量多。它是 NP 难的组合优化问题，电路布线、聚类、调度都能写成这种形式。

**思路**：变分算法把问题写成哈密顿量，用参数化线路制备试探态，再让经典优化器调整参数，使期望值越来越好。QAOA 是为组合优化设计的线路；同一套框架换一个哈密顿量，就是量子化学中的 VQE（课后作业）。

In [ ]:
from mqlab import check              # 先导入：同时屏蔽 MindSpore 的冗长日志
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import mindspore as ms
from mindquantum import QubitOperator, Hamiltonian, MaxCutAnsatz, MQAnsatzOnlyLayer
from mindquantum import Simulator, UN, Measure

## 1. 问题与经典基线

用比特串表示一种分法：第 i 位是顶点 i 所在的组，按 q4 … q0 书写，与 MindSpore Quantum 的比特序一致。5 个顶点只有 $2^5 = 32$ 种分法，可以全部枚举。有了这个经典基线，才能判断量子算法的结果好不好。

In [ ]:
graph = nx.Graph([(0, 1), (1, 2), (2, 3), (3, 4), (0, 4), (0, 2)])

def cut_size(bits):                        # 跨组的边数；bits 按 q(n−1)…q0 书写
    return sum(bits[-1 - i] != bits[-1 - j] for i, j in graph.edges)

n = graph.number_of_nodes()
max_cut = max(cut_size(f'{b:0{n}b}') for b in range(2 ** n))   # 暴力枚举
print(f'{n} 个顶点、{len(graph.edges)} 条边，最大割 = {max_cut}')

plt.figure(figsize=(3, 3))
nx.draw(graph, with_labels=True, node_color='white', edgecolors='#222', font_weight='bold')
plt.show()

## 2. QAOA：哈密顿量、线路与训练

- **哈密顿量**：对每条边 (i, j)，$Z_iZ_j$ 在两端同组时为 +1、异组时为 −1，所以 $\text{割边数} = (|E| - \langle H\rangle)/2$，其中 $H = \sum_{(i,j)\in E} Z_iZ_j$。最小化 ⟨H⟩，就是最大化割边数；
- **线路**：代价层把问题写进相位（每条边一个 ZZ 门），混合层在候选解之间“搅拌”（每个比特一个 RX 门），两层交替 p 次，共 2p 个参数。`MaxCutAnsatz` 直接生成整条线路；
- **训练**：`MQAnsatzOnlyLayer` 把“期望值 + 梯度”包装成 MindSpore 网络，用 Adam 优化器训练。

下面是完整的程序（与讲义相同），训练 200 步：

In [ ]:
p = 4                                            # QAOA 层数
ops = QubitOperator()
for i, j in graph.edges:
    ops += QubitOperator(f'Z{i} Z{j}')           # 每条边一项
ham = Hamiltonian(ops)
circ = MaxCutAnsatz(list(graph.edges), depth=p).circuit

ms.set_seed(42)
sim = Simulator('mqvector', circ.n_qubits)
grad_ops = sim.get_expectation_with_grad(ham, circ)
net = MQAnsatzOnlyLayer(grad_ops)                # 线路 → 可训练网络
opti = ms.nn.Adam(net.trainable_params(), learning_rate=0.05)
train_net = ms.nn.TrainOneStepCell(net, opti)

history = []
for step in range(200):
    cut = (len(graph.edges) - train_net()) / 2   # ⟨H⟩ → 割边数
    history.append(float(cut.asnumpy()[0]))

print(f'{len(circ.params_name)} 个参数，训练后平均割边数 {history[-1]:.3f}')
plt.figure(figsize=(6, 2.6))
plt.plot(history, color='#222')
plt.axhline(max_cut, color='#c0392b', ls='--', lw=1)   # 暴力枚举得到的最大割
plt.xlabel('step')
plt.ylabel('<cut>')
plt.show()

## 3. 结果：线路给出了哪些分法

用训练好的参数采样 1,000 次，统计每种割边数出现的频率，并与随机猜（32 种分法等概率）比较：

In [ ]:
pr = dict(zip(circ.params_name, net.weight.asnumpy()))
res = Simulator('mqvector', n).sampling(circ + UN(Measure(), n), pr=pr, shots=1000, seed=7)

qaoa = np.zeros(len(graph.edges) + 1)
for bits, k in res.data.items():
    qaoa[cut_size(bits)] += k / 1000                # QAOA：各割边数的频率
guess = np.bincount([cut_size(f'{b:0{n}b}') for b in range(2 ** n)], minlength=len(qaoa)) / 2 ** n

print('出现最多的 4 个比特串：', sorted(res.data, key=res.data.get, reverse=True)[:4])
print(f'测到最优解的概率：{qaoa[max_cut]:.1%}')
x = np.arange(len(qaoa))
plt.figure(figsize=(6, 2.6))
plt.bar(x - 0.2, guess, 0.4, color='0.75', label='random guess')
plt.bar(x + 0.2, qaoa, 0.4, color='#c0392b', label='QAOA')
plt.xlabel('cut size')
plt.ylabel('probability')
plt.legend()
plt.show()

平均割边数 4.94 < 5：线路输出的是概率分布，绝大部分概率落在最优解上。出现最多的 4 个比特串正是全部 4 个最优解（`10110` 与 `01001` 是同一种分法，只是交换了两组的名字）。

### 现场改一改：先预测，再运行

每次改一处，先写下预测，再依次运行到第 3 节：

1. **层数 1 → 4**：把第 2 节的 `p = 4` 依次改成 1、2、3。训练后的平均割边数？　A 都是 5　B 逐渐接近 5　C 都约 4.1
2. **参数全部从 0 开始**：第 2 节改成 `net = MQAnsatzOnlyLayer(grad_ops, weight='zeros')`。训练 200 步后，平均割边数？　A 照样到 4.94　B 停在 3.0 不动　C 报错
3. **换一个更大的图**：第 1 节改成 `graph = nx.random_regular_graph(3, 10, seed=1)`（10 个顶点、15 条边），从第 1 节运行起。测到最优解的概率？　A 约 96%，不变　B 约 55%　C 约 5%

<details><summary>答案</summary>

1. B：4.11、4.62、4.83、4.94。层数越多越接近 5，参数也越多（2p 个）。
2. B：停在 3.0，参数一直是 0，采样分布和随机猜一样。原点处代价层和混合层都是恒等操作，状态停在均匀叠加，两类参数的梯度恰好都为 0。默认的随机初始化正是为了离开这个点。
3. B：约 55%（5 个顶点时约 96%），平均割边数却已达到最大割的 96%。问题越大，需要的层数越多：这个图要到 p = 8 才超过 90%。
</details>

## 4. 深入：一层 QAOA 的优化地形

p = 1 时只有 γ、β 两个参数，可以把平均割边数在整个参数平面上算出来，画成“地形图”。训练就是在这张图上爬坡。

In [ ]:
c1 = MaxCutAnsatz(list(graph.edges), depth=1).circuit     # 参数：gamma_0、beta_0
f1 = Simulator('mqvector', c1.n_qubits).get_expectation_with_grad(ham, c1)
G = np.linspace(0, np.pi, 61)
B = np.linspace(-np.pi / 2, np.pi / 2, 61)
land = np.array([[(len(graph.edges) - f1(np.array([g, b]))[0].real[0, 0]) / 2 for g in G] for b in B])

plt.figure(figsize=(5, 3.4))
plt.imshow(land, origin='lower', extent=[0, np.pi, -np.pi / 2, np.pi / 2], aspect='auto',
           cmap='RdBu_r', vmin=1.5, vmax=4.5)
plt.colorbar(label='<cut>')
plt.xlabel('gamma')
plt.ylabel('beta')
plt.show()
print(f'地形最高点：平均割边数 {land.max():.2f}')

γ = 0 或 β = 0 时，平均割边数恒为 3.0，等于随机猜（图中白色）；最高点只有 4.11，所以一层不够。层数为 p 时有 2p 个参数，地形画不出来了，但平地、多个山峰这些特征依然存在：从平地出发梯度为 0（第 2 处修改），从别处出发也可能停在次优的山峰上。

## 课后作业 · 用 VQE 计算氢分子的势能曲线

VQE 与 QAOA 是同一套框架：换一个哈密顿量、换一条线路，就从组合优化走到了量子化学。

1. `get_H2_ham(d)` 生成键长为 d（单位 Å）时氢分子的量子比特哈密顿量，同时给出 FCI（精确解）与 HF（平均场近似）的能量；
2. 按下图搭建 4 比特线路，参数名依次为 `a0` … `a3`、`b0` … `b3`；
3. 对 0.5–1.1 Å 的 13 个键长分别训练，画出势能曲线，与 FCI、HF 比较。

![VQE 线路](images/vqe_ansatz.svg)

> 需要 `openfermion` 与 `openfermionpyscf`，仅支持 Linux / macOS。Windows 用户请使用 MyBinder 在线环境。

In [ ]:
import tempfile
from mindquantum import Circuit, X, RY, get_qubit_hamiltonian

def get_H2_ham(d):
    """键长 d（Å）时的氢分子哈密顿量，以及 FCI、HF 能量（Hartree）。"""
    from openfermion.chem import MolecularData
    from openfermionpyscf import run_pyscf
    mol = MolecularData([('H', (0, 0, 0)), ('H', (0, 0, d))], 'sto3g', multiplicity=1,
                        data_directory=tempfile.gettempdir())
    mol = run_pyscf(mol, run_fci=True)
    return Hamiltonian(get_qubit_hamiltonian(mol)), mol.fci_energy, mol.hf_energy

In [ ]:
ansatz = Circuit()
# 作业 1：按图搭建线路
...
check.vqe_ansatz(ansatz)

In [ ]:
distances = [round(0.5 + 0.05 * i, 2) for i in range(13)]   # 0.5–1.1 Å
energies, fci, hf = [], [], []
sim_h2 = Simulator('mqvector', 4)
for d in distances:
    ham_h2, e_fci, e_hf = get_H2_ham(d)
    fci.append(e_fci)
    hf.append(e_hf)
    # 作业 2：训练线路 200 步，把收敛后的能量加入 energies
    ...

check.vqe_curve(energies, fci)

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(distances, fci, 'k-', label='FCI')
plt.plot(distances, hf, '--', color='0.6', label='HF')
if energies:
    plt.plot(distances, energies, 'o', color='#c0392b', label='VQE')
plt.xlabel('bond length (Å)')
plt.ylabel('energy (Hartree)')
plt.legend()
plt.show()

## 小结

- 变分算法 = 参数化线路 + 经典优化器；问题的全部信息都在哈密顿量里。
- 5 个顶点的 Max-Cut 用暴力枚举瞬间可解：本节的意义在于掌握完整的变分流程，而不是加速。
- 变分算法尚未被证明能带来实用的量子优势：梯度消失、测量开销、可被经典模拟都是绕不开的问题。评价结果时，始终和经典方法比较。